# EquiSense Cross-Ticker Regime Analysis

Межтикерный анализ режимов и устойчивости признаков: сравниваем распределения и корреляции между тикерами.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="ticks", context="talk")
ROOT = Path.cwd().resolve().parents[0]
proc = ROOT / "backend" / "data" / "processed"
rows = []
for d in sorted(proc.glob("*")):
    p = d / "technical.parquet"
    if p.exists():
        t = d.name.upper()
        x = pd.read_parquet(p)
        x["date"] = pd.to_datetime(x["date"])
        x["ticker"] = t
        rows.append(x[["date", "ticker", "returns", "volatility", "rsi", "macd"]])
panel = pd.concat(rows, ignore_index=True).dropna()
summary = panel.groupby("ticker")[["returns", "volatility", "rsi"]].agg(["mean","std"])
summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.boxplot(data=panel, x="ticker", y="returns", ax=axes[0,0], palette="Set3")
axes[0,0].tick_params(axis="x", rotation=30)
axes[0,0].set_title("Returns distribution by ticker")

sns.boxplot(data=panel, x="ticker", y="volatility", ax=axes[0,1], palette="coolwarm")
axes[0,1].tick_params(axis="x", rotation=30)
axes[0,1].set_title("Volatility by ticker")

wide = panel.pivot_table(index="date", columns="ticker", values="returns").dropna(axis=1, how="any")
sns.heatmap(wide.corr(), cmap="vlag", center=0, ax=axes[1,0])
axes[1,0].set_title("Cross-ticker return correlation")

trend = panel.groupby(["date","ticker"], as_index=False)["returns"].mean()
sns.lineplot(data=trend, x="date", y="returns", hue="ticker", ax=axes[1,1], legend=False)
axes[1,1].set_title("Return regimes over time")

plt.tight_layout()

## Выводы

- Межтикерные распределения показывают, где модель должна учитывать гетерогенность активов.
- Корреляции подтверждают, что часть сигналов систематическая (фактор рынка), а часть — тикер-специфична.
- Для повышения robustness полезен подход: pooled model + ticker-level calibration.